# Análisis de Redistribución Quirúrgica
**Experimento:** ventana deslizante de 5 semanas sobre 205 iteraciones  
**Combinaciones:** 3 heurísticas × 2 estrategias × 3 niveles de exclusividad = 18 por iteración  
**Total filas:** 3 690

## 1. Configuración e importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, f_oneway, mannwhitneyu
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
OUTPUT_DIR = Path('../../output/analisis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

HEU_PAL   = {'FIFO': '#4878CF', 'SPT': '#53A76F', 'LPT': '#D65F5F'}
STRAT_PAL = {'A': '#5B8DB8', 'S': '#E87D5A'}
EXCL_PAL  = {0: '#78C1D4', 1: '#F4A259', 2: '#C87BB7'}
EXCL_LABEL = {0: '0 excl.', 1: '1 excl.', 2: '2 excl.'}
HEU_ORDER  = ['SPT', 'FIFO', 'LPT']

print('Librerias cargadas correctamente.')

## 2. Carga y preprocesamiento de datos

In [ ]:
df = pd.read_csv('../../data/resultados_experimento.csv')

weeks_df  = pd.read_csv('../../data/semanas_ordenadas.csv')
weeks_vol = weeks_df.set_index('weekStart')['totalSurgeries']

for k in ['semana_1', 'semana_2', 'semana_3', 'semana_4', 'semana_5']:
    df[f'vol_{k}'] = df[k].map(weeks_vol)

df['vol_ventana'] = df[['vol_semana_1','vol_semana_2','vol_semana_3',
                          'vol_semana_4','vol_semana_5']].sum(axis=1)
df['excl_label'] = df['salas_exclusivas'].map(EXCL_LABEL)
df['heu_strat']  = df['heuristica'] + '+' + df['estrategia']

print(f"Filas: {len(df):,}  |  Iteraciones: {df['iteracion'].nunique()}  |  Combinaciones: {df.groupby(['heuristica','estrategia','salas_exclusivas']).ngroups}")
df.head(3)

## 3. Estadísticas descriptivas generales

In [ ]:
desc = (
    df.groupby(['heuristica', 'estrategia', 'salas_exclusivas'])
    [['throughput', 'lead_time', 'util_urg_pct', 'util_elec_pct', 'util_global_pct']]
    .agg(['mean', 'std', 'median', 'min', 'max'])
    .round(2)
)
desc

> **Lectura de la tabla.** Cada fila corresponde a una de las 18 combinaciones evaluadas sobre 205 iteraciones independientes. Los valores de `mean` y `std` son los de referencia para el análisis comparativo; la `median` es preferible como medida de tendencia central cuando la distribución presenta asimetría, como ocurre con LPT y en menor medida con FIFO. Nótese que SPT presenta la desviación estándar más baja en throughput, lo que anticipa una mayor estabilidad entre iteraciones.

## 4. Throughput
### 4.1 Distribución general por heurística

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(
    data=df, x='heuristica', y='throughput',
    order=HEU_ORDER, palette=HEU_PAL,
    inner='box', linewidth=1.2, ax=ax
)

means = df.groupby('heuristica')['throughput'].mean()
for i, h in enumerate(HEU_ORDER):
    ax.scatter(i, means[h], color='white', edgecolors='black',
               s=80, zorder=5, linewidths=1.5)
    ax.text(i + 0.15, means[h], f'{means[h]:.0f}',
            va='center', fontsize=10, fontweight='bold')

ax.set_title('Throughput Distribution by Heuristic (all iterations and exclusivity levels)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Heuristic')
ax.set_ylabel('Throughput (completed surgeries)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_violin_throughput.png', bbox_inches='tight')
plt.show()

> **Interpretación.** La diferencia entre heurísticas es contundente: SPT supera en media a FIFO en un factor de ~2.6x (533 vs 206 cirugías) y a LPT en ~5.6x (533 vs 95). Más relevante que la media es la **forma de la distribución**: SPT muestra un violín compacto y simétrico, indicando alta consistencia entre iteraciones. FIFO presenta mayor dispersión (la cola superior alcanza ~440 cirugías), mientras que LPT exhibe la distribución más asimétrica y con mayor varianza relativa, lo que revela inestabilidad dependiente de la composición de la semana.

### 4.2 Throughput por heurística, estrategia y exclusividad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, excl in zip(axes, [0, 1, 2]):
    sub = df[df['salas_exclusivas'] == excl]
    sns.boxplot(
        data=sub, x='heuristica', y='throughput', hue='estrategia',
        palette=STRAT_PAL, order=HEU_ORDER,
        ax=ax, width=0.55, linewidth=1.3, fliersize=2
    )
    medias = sub.groupby(['heuristica','estrategia'])['throughput'].mean().reset_index()
    offsets = {'A': -0.2, 'S': 0.2}
    for _, row in medias.iterrows():
        x_pos = HEU_ORDER.index(row['heuristica']) + offsets[row['estrategia']]
        ax.scatter(x_pos, row['throughput'], color='white',
                   edgecolors='black', s=50, zorder=5, linewidths=1.2)
    ax.set_title(f'{excl} exclusive operating rooms', fontsize=12, fontweight='bold')
    ax.set_xlabel('Heuristic')
    ax.set_ylabel('Throughput' if excl == 0 else '')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, ['Alternating (V)', 'Sequential (H)'], title='Strategy', fontsize=9)

fig.suptitle('Throughput by Heuristic, Strategy, and Exclusivity Level',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_throughput_boxplot.png', bbox_inches='tight')
plt.show()

> **Interpretación.** Al desagregar por nivel de exclusividad, se confirma que la ventaja de SPT se mantiene independientemente del número de pabellones reservados para urgencias. La estrategia Secuencial (H) supera consistentemente a la Alternada (V) en SPT y FIFO, aunque la diferencia es pequeña (~13 cirugías en SPT con 0 exclusivos). Un hallazgo relevante es que la superioridad de SPT se **comprime** al aumentar la exclusividad —lo que sugiere que la restricción de capacidad electiva afecta proporcionalmente más a heurísticas de alta demanda—, aunque incluso con 2 pabellones exclusivos, SPT sigue siendo la mejor opción por un amplio margen.

### 4.3 Efecto del nivel de exclusividad sobre el throughput

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, heu in zip(axes, HEU_ORDER):
    sub = df[df['heuristica'] == heu]
    sns.boxplot(
        data=sub, x='excl_label', y='throughput', hue='estrategia',
        palette=STRAT_PAL, order=['0 excl.','1 excl.','2 excl.'],
        ax=ax, width=0.55, linewidth=1.3, fliersize=2
    )
    ax.set_title(f'Heuristic: {heu}', fontsize=12, fontweight='bold',
                 color=HEU_PAL[heu])
    ax.set_xlabel('Exclusivity')
    ax.set_ylabel('Throughput' if heu == HEU_ORDER[0] else '')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, ['Alternating', 'Sequential'], title='Strategy', fontsize=9)

fig.suptitle('Effect of Exclusivity Level on Throughput by Heuristic',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_throughput_exclusividad.png', bbox_inches='tight')
plt.show()

> **Interpretación.** El costo de reservar pabellones para urgencias es diferente según la heurística: pasar de 0 a 2 pabellones exclusivos reduce el throughput en **6.6% para SPT**, **16.1% para FIFO** y **22.3% para LPT**. Esto implica que SPT es la heurística más **robusta a la política de exclusividad**: aunque se reduzca la capacidad electiva disponible, su capacidad de empaquetar cirugías cortas amortigua el impacto. FIFO y LPT son más sensibles a esta restricción porque no aprovechan los slots disponibles con igual eficiencia.

## 5. Lead Time
### 5.1 Distribución general por heurística

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(
    data=df, x='heuristica', y='lead_time',
    order=HEU_ORDER, palette=HEU_PAL,
    inner='box', linewidth=1.2, ax=ax
)

means = df.groupby('heuristica')['lead_time'].mean()
for i, h in enumerate(HEU_ORDER):
    ax.scatter(i, means[h], color='white', edgecolors='black',
               s=80, zorder=5, linewidths=1.5)
    ax.text(i + 0.15, means[h], f'{means[h]:.2f}d',
            va='center', fontsize=10, fontweight='bold')

ax.set_title('Lead Time Distribution by Heuristic',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Heuristic')
ax.set_ylabel('Lead Time (days)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_violin_leadtime.png', bbox_inches='tight')
plt.show()

> **Interpretación — advertencia sobre LPT.** A primera vista LPT parece competitivo en lead time (media 1.50 días, similar a SPT con 1.54 días). Sin embargo, esto es un **artefacto de selección**: LPT solo atiende ~95 cirugías por iteración (las más largas, que son prioritarias en su cola), dejando sin programar la gran mayoría de la demanda. El lead time calculado corresponde exclusivamente a ese subconjunto privilegiado. Las cirugías no completadas —que representan el grueso de la lista de espera— no contribuyen al promedio, lo que infla artificialmente la métrica. **LPT no debe interpretarse como eficiente en tiempos de espera; es simplemente el efecto de atender muy pocos pacientes.**
>
> SPT, en cambio, combina un lead time bajo (1.54 días en media) con un volumen de atención ~5.6 veces superior, lo que lo convierte en la única heurística con ventaja real en ambas dimensiones.

### 5.2 Lead Time por heurística, estrategia y exclusividad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, excl in zip(axes, [0, 1, 2]):
    sub = df[df['salas_exclusivas'] == excl]
    sns.boxplot(
        data=sub, x='heuristica', y='lead_time', hue='estrategia',
        palette=STRAT_PAL, order=HEU_ORDER,
        ax=ax, width=0.55, linewidth=1.3, fliersize=2
    )
    ax.set_title(f'{excl} exclusive operating rooms', fontsize=12, fontweight='bold')
    ax.set_xlabel('Heuristic')
    ax.set_ylabel('Lead Time (days)' if excl == 0 else '')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, ['Alternating (V)', 'Sequential (H)'], title='Strategy', fontsize=9)

fig.suptitle('Lead Time by Heuristic, Strategy, and Exclusivity Level',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_leadtime_boxplot.png', bbox_inches='tight')
plt.show()

> **Interpretación.** El lead time de SPT se mantiene estable y bajo (~1.5 días) en todos los niveles de exclusividad, lo que refleja que el sistema es capaz de procesar rápidamente incluso con menos pabellones electivos disponibles. FIFO muestra mayor varianza y una mediana ligeramente superior (~2.2 días), lo cual es esperable dado que no discrimina por duración y puede encolar cirugías largas que bloquean slots. La estrategia (Alternada vs Secuencial) tiene un efecto marginal sobre el lead time en comparación con la elección de heurística.

## 6. Utilización
### 6.1 Heatmaps de media de utilización

In [ ]:
metrics_info = [
    ('util_elec_pct',   'Elective Utilization (%)',   'YlOrRd'),
    ('util_urg_pct',    'Emergency Utilization (%)',  'Blues'),
    ('util_global_pct', 'Global Utilization (%)',     'YlGn'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, (metric, title, cmap) in zip(axes, metrics_info):
    pivot = (
        df.groupby(['tipo_asignacion', 'salas_exclusivas'])[metric]
        .mean()
        .unstack()
        .rename(columns=EXCL_LABEL)
    )
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
    sns.heatmap(
        pivot.round(1), annot=True, fmt='.1f', cmap=cmap,
        linewidths=0.5, ax=ax, cbar_kws={'label': '%'}, vmin=0
    )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Exclusive operating rooms')
    ax.set_ylabel('Combination' if ax == axes[0] else '')
    ax.tick_params(axis='x', labelsize=10)
    ax.tick_params(axis='y', labelsize=9)

fig.suptitle('Mean Utilization by Combination and Exclusivity Level',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_heatmaps_utilizacion.png', bbox_inches='tight')
plt.show()

> **Interpretación.** Los heatmaps revelan tres patrones estructurales importantes:
> - **Utilización electiva:** SPT alcanza valores cercanos o superiores al 100% en 0 exclusivos, confirmando que el turno de pabellón se usa casi en su totalidad. Al aumentar la exclusividad, la utilización electiva *sube* —paradójicamente— porque los pabellones electivos que quedan se saturan con la misma demanda distribuida en menos unidades.
> - **Utilización de urgencias:** Es independiente de la heurística electiva (el valor es el mismo para toda la fila), lo que confirma que la lógica de urgencias opera de forma desacoplada. El valor bajo (~26%) refleja que la demanda de urgencias no llena la capacidad de un pabellón dedicado 24/7.
> - **Utilización global:** Cae al aumentar la exclusividad porque el denominador (capacidad total) crece al sumar pabellones de urgencia, pero su contribución efectiva es baja dado el volumen real de urgencias.

### 6.2 Utilización electiva por estrategia

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, excl in zip(axes, [0, 1, 2]):
    sub = df[df['salas_exclusivas'] == excl]
    sns.boxplot(
        data=sub, x='heuristica', y='util_elec_pct', hue='estrategia',
        palette=STRAT_PAL, order=HEU_ORDER,
        ax=ax, width=0.55, linewidth=1.3, fliersize=2
    )
    ax.axhline(100, color='red', linestyle='--', linewidth=1, alpha=0.7, label='100%')
    ax.set_title(f'{excl} exclusive operating rooms', fontsize=12, fontweight='bold')
    ax.set_xlabel('Heuristic')
    ax.set_ylabel('Elective Utilization (%)' if excl == 0 else '')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, ['Alternating', 'Sequential', '100%'], title='Strategy', fontsize=9)

fig.suptitle('Elective Operating Room Utilization by Heuristic and Exclusivity',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_utilizacion_electivos.png', bbox_inches='tight')
plt.show()

> **Interpretación.** La línea roja punteada marca el 100% de utilización del turno electivo (incluyendo el delta de overflow). SPT+Secuencial supera ese umbral en varios escenarios, lo que indica que en semanas de alto volumen el sistema necesita el tiempo extra permitido. Esto es operacionalmente relevante: sugiere que con SPT el hospital operaría sistemáticamente cerca de su límite de capacidad diaria. FIFO y LPT se mantienen por debajo, no por eficiencia, sino porque no completan suficiente volumen para saturar el turno.

## 7. Throughput vs Lead Time — Frontera de Pareto

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True, sharex=False)

for ax, excl in zip(axes, [0, 1, 2]):
    sub = df[df['salas_exclusivas'] == excl]
    for heu in HEU_ORDER:
        grp = sub[sub['heuristica'] == heu]
        ax.scatter(grp['throughput'], grp['lead_time'],
                   color=HEU_PAL[heu], alpha=0.18, s=12, label=heu)
    for (heu, strat), grp in sub.groupby(['heuristica', 'estrategia']):
        mx, my = grp['throughput'].mean(), grp['lead_time'].mean()
        marker = 'D' if strat == 'A' else 's'
        ax.scatter(mx, my, color=HEU_PAL[heu], s=120, marker=marker,
                   edgecolors='black', linewidths=1, zorder=6)
        ax.annotate(f'{heu}+{strat}', (mx, my),
                    textcoords='offset points', xytext=(4, 4), fontsize=8)
    ax.set_title(f'{excl} exclusive operating rooms', fontsize=12, fontweight='bold')
    ax.set_xlabel('Throughput (surgeries)')
    if excl == 0:
        ax.set_ylabel('Lead Time (days)')

legend_handles = [
    mpatches.Patch(color=c, label=h) for h, c in HEU_PAL.items()
] + [
    plt.scatter([], [], marker='D', color='gray', s=60, label='Alternating (D)'),
    plt.scatter([], [], marker='s', color='gray', s=60, label='Sequential (S)'),
]
fig.legend(handles=legend_handles, loc='upper right',
           bbox_to_anchor=(1.01, 0.95), fontsize=9)
fig.suptitle('Throughput vs Lead Time — Means by Combination (large points)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_pareto_tp_lt.png', bbox_inches='tight')
plt.show()

> **Interpretación.** Este gráfico es el más informativo para la toma de decisión: en el espacio throughput-lead time, la **esquina óptima es arriba a la izquierda** (muchas cirugías, poco tiempo de espera). SPT ocupa esa posición dominante en los tres niveles de exclusividad. FIFO se sitúa en la zona media-baja con lead times similares a SPT pero con una fracción del volumen. LPT queda aislado en la esquina de bajo throughput, lejos de cualquier frontera eficiente.
>
> La nube de puntos pequeños (observaciones individuales) muestra que la superioridad de SPT es **consistente iteración a iteración**, no solo en promedio: su cluster está sistemáticamente separado del de FIFO y LPT. No existe ninguna iteración en que LPT supere a SPT en throughput.

## 8. Tests estadísticos
### 8.1 Kruskal-Wallis y ANOVA — efecto de la heurística

In [ ]:
print(f"{'='*70}")
print(f"  TESTS ESTADISTICOS — EFECTO DE LA HEURISTICA")
print(f"{'='*70}")

for excl in [0, 1, 2]:
    sub = df[df['salas_exclusivas'] == excl]
    print(f"\n--- Salas exclusivas = {excl} ---")
    print(f"  {'Metrica':<22s}  {'KW H':>8}  {'KW p':>10}  {'ANOVA F':>9}  {'ANOVA p':>10}  Sig")
    print(f"  {'-'*65}")
    for metric in ['throughput', 'lead_time', 'util_elec_pct', 'util_global_pct']:
        groups = [sub[sub['heuristica'] == h][metric].values for h in ['FIFO','SPT','LPT']]
        H, p_kw = kruskal(*groups)
        F, p_an = f_oneway(*groups)
        p_min = min(p_kw, p_an)
        sig = '***' if p_min < 0.001 else '**' if p_min < 0.01 else '*' if p_min < 0.05 else 'ns'
        print(f"  {metric:<22s}  {H:>8.1f}  {p_kw:>10.2e}  {F:>9.1f}  {p_an:>10.2e}  {sig}")

print("\n*** p<0.001  ** p<0.01  * p<0.05  ns = no significativo")

> **Interpretación.** Los tests confirman que las diferencias entre heurísticas son **estadísticamente significativas (p < 0.001)** para throughput en los tres niveles de exclusividad, tanto en el test no paramétrico de Kruskal-Wallis como en ANOVA. Esto descarta que la superioridad de SPT sea un artefacto del muestreo: con 205 observaciones por combinación, los resultados tienen alta potencia estadística. El efecto sobre lead time también es significativo, aunque —como se discutió— debe interpretarse con cautela para LPT.

### 8.2 Efecto de la exclusividad y la estrategia

In [ ]:
print(f"{'='*60}")
print(f"  EFECTO DE EXCLUSIVIDAD Y ESTRATEGIA (Kruskal-Wallis)")
print(f"{'='*60}")

for metric in ['throughput', 'lead_time', 'util_elec_pct']:
    print(f"\nMetrica: {metric}")
    groups_excl = [df[df['salas_exclusivas'] == e][metric].values for e in [0,1,2]]
    H, p = kruskal(*groups_excl)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  Exclusividad (0 vs 1 vs 2) : H={H:.1f}  p={p:.2e}  {sig}")
    groups_strat = [df[df['estrategia'] == s][metric].values for s in ['A','S']]
    H, p = kruskal(*groups_strat)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"  Estrategia (A vs S)        : H={H:.1f}  p={p:.2e}  {sig}")

> **Interpretación.** Tanto la exclusividad como la estrategia tienen efectos estadísticamente significativos, aunque de **menor magnitud práctica** que la heurística. La estrategia Secuencial supera a la Alternada en throughput (p < 0.001), pero la diferencia media es de ~11 cirugías por ventana (~4%), lo que puede no ser clínicamente relevante en todos los contextos. El efecto de la exclusividad es más pronunciado porque modifica directamente la capacidad disponible para electivas.

### 8.3 Comparación pareada post-hoc (Mann-Whitney + Bonferroni)

In [ ]:
print(f"{'='*70}")
print(f"  COMPARACION PAREADA — Throughput (Mann-Whitney + Bonferroni)")
print(f"{'='*70}")

n_comp = 3
for excl in [0, 1, 2]:
    sub = df[df['salas_exclusivas'] == excl]
    print(f"\n--- Salas exclusivas = {excl} ---")
    for h1, h2 in combinations(['FIFO','SPT','LPT'], 2):
        g1 = sub[sub['heuristica'] == h1]['throughput'].values
        g2 = sub[sub['heuristica'] == h2]['throughput'].values
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
        p_bonf = min(p * n_comp, 1.0)
        sig = '***' if p_bonf < 0.001 else '**' if p_bonf < 0.01 else '*' if p_bonf < 0.05 else 'ns'
        med1, med2 = np.median(g1), np.median(g2)
        print(f"  {h1} vs {h2}:  mediana {med1:.0f} vs {med2:.0f}  |  p_bonf={p_bonf:.2e}  {sig}")

print("\n*** p<0.001  ** p<0.01  * p<0.05  ns = no significativo")

> **Interpretación.** Los tres pares de heurísticas presentan diferencias significativas (p < 0.001) en todos los niveles de exclusividad, incluso tras aplicar corrección de Bonferroni para comparaciones múltiples. Esto confirma que los tres algoritmos corresponden a **clases de rendimiento estadísticamente distinguibles**, no a variantes de un mismo comportamiento. Para el paper, este resultado permite afirmar con rigor estadístico que la elección de heurística es el factor de diseño más determinante del sistema.

## 9. Correlación con el volumen de la ventana

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)

for row_i, metric in enumerate(['throughput', 'lead_time']):
    for col_i, excl in enumerate([0, 1, 2]):
        ax = axes[row_i, col_i]
        sub = df[df['salas_exclusivas'] == excl]
        for heu in HEU_ORDER:
            grp = sub[sub['heuristica'] == heu]
            ax.scatter(grp['vol_ventana'], grp[metric],
                       color=HEU_PAL[heu], alpha=0.25, s=10, label=heu)
        x = sub['vol_ventana'].values
        y = sub[metric].values
        slope, intercept, r, p, _ = stats.linregress(x, y)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, slope * x_line + intercept,
                color='black', linewidth=1.5, linestyle='--', zorder=5)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        ax.set_title(f'{excl} excl. | r={r:.2f} ({sig})', fontsize=11, fontweight='bold')
        ax.set_xlabel('Window volume (total surgeries)')
        if col_i == 0:
            ax.set_ylabel(metric.replace('_', ' ').title())
        if row_i == 0 and col_i == 0:
            ax.legend(title='Heuristic', markerscale=2, fontsize=9)

fig.suptitle('Correlation Between Window Volume and Performance Metrics',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_correlacion_volumen.png', bbox_inches='tight')
plt.show()

In [ ]:
print(f"{'='*65}")
print(f"  CORRELACIONES DE PEARSON: vol_ventana vs metricas")
print(f"{'='*65}")
print(f"  {'Combinacion':<22}  {'r_TP':>7}  {'r_LT':>7}  {'r_uElec':>8}  {'r_uGlob':>8}")
print(f"  {'-'*55}")
for combo, grp in df.groupby('tipo_asignacion'):
    r_tp, _ = stats.pearsonr(grp['vol_ventana'], grp['throughput'])
    r_lt, _ = stats.pearsonr(grp['vol_ventana'], grp['lead_time'])
    r_ue, _ = stats.pearsonr(grp['vol_ventana'], grp['util_elec_pct'])
    r_ug, _ = stats.pearsonr(grp['vol_ventana'], grp['util_global_pct'])
    print(f"  {combo:<22}  {r_tp:>7.3f}  {r_lt:>7.3f}  {r_ue:>8.3f}  {r_ug:>8.3f}")

> **Interpretación.** Este es uno de los hallazgos más relevantes del experimento: **SPT presenta una correlación casi perfecta entre volumen de la ventana y throughput (r ≈ 0.97)**, lo que indica que la heurística escala de forma casi lineal con la demanda. El sistema responde proporcionalmente a la carga: más demanda se traduce en más cirugías completadas.
>
> En contraste, **FIFO muestra correlación cercana a cero (r ≈ −0.02)**: su throughput no crece con el volumen disponible, indicando que la heurística alcanza un techo de capacidad rápidamente independientemente de cuánta demanda haya. Esto sugiere que FIFO desperdicia capacidad cuando el volumen es alto.
>
> La correlación entre volumen y lead time es despreciable (r ≈ −0.02) para todas las combinaciones, lo que implica que el **lead time no depende del volumen total de la semana**, sino de la lógica interna de asignación. Este resultado es valioso para el paper: confirma que mejorar el lead time es principalmente un problema de diseño de la política de scheduling, no de capacidad hospitalaria.

## 10. ¿Qué combinación gana más iteraciones?
### 10.1 Mayor throughput por iteración

In [ ]:
winner_tp = (
    df.sort_values(['iteracion','throughput'], ascending=[True, False])
    .groupby('iteracion').first().reset_index()
)
wins_tp = (
    winner_tp.groupby(['tipo_asignacion','salas_exclusivas'])
    .size().reset_index(name='wins').sort_values('wins', ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = [HEU_PAL[r.split(' + ')[0]] for r in wins_tp['tipo_asignacion']]
bars = ax.bar(range(len(wins_tp)), wins_tp['wins'],
              color=bar_colors, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(len(wins_tp)))
ax.set_xticklabels(
    [f"{row['tipo_asignacion']}\nexcl={row['salas_exclusivas']}" for _, row in wins_tp.iterrows()],
    rotation=30, ha='right', fontsize=9
)
ax.set_ylabel('Iterations won (maximum throughput)')
ax.set_title('Configuration with Highest Throughput per Iteration', fontsize=13, fontweight='bold')
for bar, val in zip(bars, wins_tp['wins']):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(val)), ha='center', va='bottom', fontsize=9, fontweight='bold')
legend_h = [mpatches.Patch(color=c, label=h) for h, c in HEU_PAL.items()]
ax.legend(handles=legend_h, title='Heuristic')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_wins_throughput.png', bbox_inches='tight')
plt.show()

> **Interpretación.** SPT gana en throughput en **191 de 205 iteraciones (93%)**. Las 14 iteraciones restantes corresponden a ventanas de muy bajo volumen donde la diferencia entre heurísticas se diluye. La consistencia de SPT como mejor opción es, desde el punto de vista estadístico y práctico, robusta. Esta métrica —número de victorias por iteración— es especialmente útil para el paper porque refleja la **dominancia estocástica** de SPT: no solo es mejor en promedio, sino que es mejor casi siempre.

### 10.2 Menor lead time por iteración

In [ ]:
winner_lt = (
    df.sort_values(['iteracion','lead_time'], ascending=[True, True])
    .groupby('iteracion').first().reset_index()
)
wins_lt = (
    winner_lt.groupby(['tipo_asignacion','salas_exclusivas'])
    .size().reset_index(name='wins').sort_values('wins', ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = [HEU_PAL[r.split(' + ')[0]] for r in wins_lt['tipo_asignacion']]
bars = ax.bar(range(len(wins_lt)), wins_lt['wins'],
              color=bar_colors, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(len(wins_lt)))
ax.set_xticklabels(
    [f"{row['tipo_asignacion']}\nexcl={row['salas_exclusivas']}" for _, row in wins_lt.iterrows()],
    rotation=30, ha='right', fontsize=9
)
ax.set_ylabel('Iterations won (lowest lead time)')
ax.set_title('Configuration with Lowest Lead Time per Iteration', fontsize=13, fontweight='bold')
for bar, val in zip(bars, wins_lt['wins']):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(int(val)), ha='center', va='bottom', fontsize=9, fontweight='bold')
legend_h = [mpatches.Patch(color=c, label=h) for h, c in HEU_PAL.items()]
ax.legend(handles=legend_h, title='Heuristic')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_wins_leadtime.png', bbox_inches='tight')
plt.show()

> **Interpretación.** El ranking de victorias en lead time debe leerse con la precaución discutida anteriormente: LPT puede aparecer como ganador en muchas iteraciones precisamente porque atiende muy pocas cirugías (las más largas, que son también las que más rápido se programan al estar al inicio de la cola). Este resultado **no debe interpretarse como una ventaja clínica real de LPT** para los tiempos de espera del paciente. La comparación de lead time solo tiene validez cuando el throughput es comparable, lo cual ocurre únicamente entre SPT y FIFO.

## 11. Evolución por iteración — Estabilidad temporal

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

for ax, metric, ylabel in zip(
    axes,
    ['throughput', 'lead_time'],
    ['Throughput (surgeries)', 'Lead Time (days)']
):
    for heu in HEU_ORDER:
        for strat, ls in [('A', '--'), ('S', '-')]:
            sub = df[(df['heuristica'] == heu) &
                     (df['estrategia'] == strat) &
                     (df['salas_exclusivas'] == 0)]
            med_iter = sub.groupby('iteracion')[metric].median()
            ax.plot(med_iter.index, med_iter.values,
                    color=HEU_PAL[heu], linestyle=ls,
                    linewidth=1, alpha=0.85, label=f'{heu}+{strat}')
    ax.set_ylabel(ylabel)
    ax.legend(ncol=3, fontsize=9, loc='upper right')

axes[1].set_xlabel('Iteration (window)')
fig.suptitle('Metrics Evolution by Iteration (excl=0, median)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_evolucion_iteraciones.png', bbox_inches='tight')
plt.show()

> **Interpretación.** La evolución temporal confirma la **estabilidad de los resultados**: las curvas de cada heurística se mantienen bien separadas y con tendencia estable a lo largo de las 205 iteraciones. No se observan cruces entre SPT y las otras heurísticas, lo que refuerza la conclusión de dominancia. La variación entre iteraciones (rango vertical de cada curva) refleja el efecto del volumen de la ventana, lo que es coherente con la alta correlación r=0.97 identificada para SPT. Las iteraciones con menor throughput corresponden a ventanas de semanas menos cargadas del dataset.

## 12. Tabla resumen ejecutiva para el paper

In [ ]:
summary_paper = (
    df.groupby(['tipo_asignacion', 'salas_exclusivas'])
    .agg(
        TP_mean    = ('throughput',      'mean'),
        TP_std     = ('throughput',      'std'),
        TP_median  = ('throughput',      'median'),
        LT_mean    = ('lead_time',       'mean'),
        LT_std     = ('lead_time',       'std'),
        uElec_mean = ('util_elec_pct',   'mean'),
        uElec_std  = ('util_elec_pct',   'std'),
        uGlob_mean = ('util_global_pct', 'mean'),
        uGlob_std  = ('util_global_pct', 'std'),
    )
    .round(2)
    .sort_values('TP_mean', ascending=False)
)

summary_paper.to_csv('tabla_resumen_paper.csv')
print('Tabla exportada a tabla_resumen_paper.csv')
display(summary_paper)

In [ ]:
print(f"\n{'Combinacion':<22} {'Excl':>5} | {'TP mean+/-std':>16} | {'LT mean+/-std':>14} | {'uElec%':>8} | {'uGlob%':>8}")
print('-' * 82)
for (combo, excl), row in summary_paper.iterrows():
    print(
        f"{combo:<22} {excl:>5} | "
        f"{row['TP_mean']:>6.0f} +/- {row['TP_std']:>5.0f} | "
        f"{row['LT_mean']:>5.2f} +/- {row['LT_std']:>4.2f} | "
        f"{row['uElec_mean']:>7.1f}% | "
        f"{row['uGlob_mean']:>7.1f}%"
    )

---
## 13. Conclusiones

### 13.1 Ranking de las tres mejores configuraciones

El análisis sobre 205 iteraciones y 3 690 escenarios permite establecer con solidez estadística las siguientes conclusiones:

---

#### 🥇 Configuración 1 — SPT + Secuencial (Horizontal) + 0 pabellones exclusivos

| Métrica | Valor |
|---|---|
| Throughput (media ± std) | 557 ± 159 cirugías |
| Lead Time (media) | 1.52 días |
| Utilización electiva | ~100% |
| Victorias en throughput | 93% de las iteraciones |

**Justificación.** SPT (*Shortest Processing Time*) maximiza el throughput al priorizar las cirugías de menor duración, lo que permite encadenar mayor cantidad de intervenciones dentro del turno disponible. La estrategia Secuencial (Horizontal) complementa este efecto al concentrar la demanda en un mismo pabellón antes de abrir el siguiente, minimizando los tiempos muertos de preparación entre cirugías. La ausencia de pabellones exclusivos de urgencias añade flexibilidad total: cualquier pabellón libre puede absorber una urgencia, reduciendo el costo de oportunidad de la exclusividad sin comprometer la respuesta de emergencia. La correlación casi perfecta de SPT con el volumen (r = 0.97) indica además que esta configuración **escala linealmente con la demanda**, una propiedad deseable para su implementación en semanas de alta presión asistencial.

---

#### 🥈 Configuración 2 — SPT + Alternado (Vertical) + 0 pabellones exclusivos

| Métrica | Valor |
|---|---|
| Throughput (media ± std) | 543 ± 153 cirugías |
| Lead Time (media) | 1.51 días |
| Utilización electiva | ~94% |

**Justificación.** Prácticamente idéntica a la configuración 1 en lead time y con una reducción marginal de throughput (~2.5%). La estrategia Alternada (Vertical) distribuye las cirugías entre pabellones en lugar de encadenarlas, lo que puede resultar operacionalmente más robusta ante interrupciones inesperadas (equipos, personal) ya que no genera dependencias largas en un solo pabellón. Es la opción recomendada cuando la gestión operacional de un pabellón completamente ocupado es difícil, o cuando se busca mayor **flexibilidad intradía** para reasignar urgencias sin interrumpir cadenas de cirugías en curso.

---

#### 🥉 Configuración 3 — FIFO + Secuencial (Horizontal) + 0 pabellones exclusivos

| Métrica | Valor |
|---|---|
| Throughput (media ± std) | 244 ± 136 cirugías |
| Lead Time (media) | 1.69 días |
| Utilización electiva | ~67% |

**Justificación.** FIFO (*First In, First Out*) es la única heurística evaluada que respeta el principio de **equidad temporal de acceso**: los pacientes son atendidos en el orden en que ingresaron a la lista de espera, independientemente de la complejidad o duración de su intervención. Aunque su throughput es significativamente inferior al de SPT (~56% menos), este criterio tiene relevancia clínica y ética en sistemas de salud donde la percepción de justicia del paciente es un indicador de calidad. FIFO es la opción recomendada cuando el hospital tiene compromisos institucionales de tiempo máximo de espera o cuando la lista de espera incluye pacientes con urgencia relativa que no debe ser pospuesta por cirugías más cortas.

---

### 13.2 Sobre LPT y por qué no se recomienda

LPT (*Longest Processing Time*) exhibe un throughput promedio de solo 95 cirugías (~18% del de SPT) y, aunque sus estadísticos de lead time aparentan competitividad, esto es un **artefacto de selección**: LPT únicamente completa las cirugías más largas —que son pocas y están al inicio de su cola—, dejando sin atender la gran mayoría de la demanda. En ninguna de las 205 iteraciones LPT superó a SPT en throughput. Su uso implicaría una subutilización severa de la capacidad quirúrgica y no es recomendable en ningún escenario clínico analizado.

---

### 13.3 Sobre el nivel de exclusividad

Desde el punto de vista del throughput, **0 pabellones exclusivos domina consistentemente**. Sin embargo, la elección del nivel de exclusividad involucra consideraciones operacionales no capturadas por el modelo (tiempos de preparación específicos para urgencias, disponibilidad de equipamiento especializado, normativa hospitalaria), por lo que los resultados de exclusividad 1 y 2 se presentan como alternativas válidas cuando existan restricciones institucionales.

---

### 13.4 Limitaciones del estudio

- Las ventanas son **solapadas** (comparten 4 de 5 semanas entre iteraciones consecutivas), lo que introduce correlación serial entre observaciones. Se recomienda reportar este diseño explícitamente en el paper y complementar con un análisis de sensibilidad sobre ventanas no solapadas.
- El modelo no considera **tiempos de setup** entre cirugías, disponibilidad de personal ni especialidades quirúrgicas.
- El lead time calculado es desde el *inicio simulado* de la demanda, no desde la fecha real de solicitud del paciente.